# HR snapshot cycle

One policy, in order: create, edit, ACL change, delete. Each cell prints a
row of source state against the S3 snapshot. After the delete sweep, the
snapshot count matches `/ids` again (drift 0).

This is the notebook to run live. `make e2e` is the same cycle in pytest and
is not part of `make test`. Create, edit, and ACL usually land within one
10 second poll. The delete waits for the 60 second reconcile sweep.


In [1]:
import sys
from pathlib import Path

here = Path.cwd()
if not (here / "hr_demo.py").exists():
    here = here / "notebooks"
if str(here) not in sys.path:
    sys.path.insert(0, str(here))

import hr_demo


## Precondition


In [2]:
POLICY_ID = "pol-demo-cycle"
KEY = f"raw/hr/policy/{POLICY_ID}.md"
rows = ["step              source                      snapshot"]
hr_demo.delete_if_present("policy", POLICY_ID)

def show(step: str) -> None:
    line = hr_demo.describe(step, "policy", POLICY_ID)
    rows.append(line)
    print(line)


## Create


In [3]:
created = hr_demo.admin(
    "POST",
    "/admin/policies",
    {
        "policy_id": POLICY_ID,
        "title": "Demo cycle",
        "category": "leave",
        "region": "DE",
        "format": "markdown",
        "native_acl": ["hr_employee"],
        "body": "# Demo cycle\n\nCreated by the notebook.\n",
    },
)
hr_demo.wait_until(
    lambda: hr_demo.snapshot(KEY) is not None
    and b"Created by the notebook" in hr_demo.object_body(KEY),
    hr_demo.WRITE_WAIT,
    "create snapshot",
)
show("create")


create            v1 hr_employee              v1 hr_employee versions=1


## Update


In [4]:
before = hr_demo.versions(KEY)
edited = hr_demo.admin(
    "PUT",
    f"/admin/policies/{POLICY_ID}",
    {"body": "# Demo cycle\n\nEdited by the notebook.\n"},
)
assert edited["version"] == created["version"] + 1
hr_demo.wait_until(
    lambda: hr_demo.versions(KEY) == before + 1
    and b"Edited by the notebook" in hr_demo.object_body(KEY),
    hr_demo.WRITE_WAIT,
    "policy rewrite",
)
show("update")


update            v2 hr_employee              v2 hr_employee versions=2


## ACL change


In [5]:
before_hash = hr_demo.snapshot(KEY)["Metadata"][hr_demo.META_HASH]
before_version = hr_demo.snapshot(KEY)["Metadata"][hr_demo.META_VERSION]
changed = hr_demo.admin(
    "PUT",
    f"/admin/policies/{POLICY_ID}/acl",
    {"native_acl": ["hr_policy_admin"]},
)
assert changed["version"] == int(before_version)

def acl_written() -> bool:
    meta = (hr_demo.snapshot(KEY) or {}).get("Metadata", {})
    return (
        meta.get(hr_demo.META_ACL) == "hr_policy_admin"
        and meta.get(hr_demo.META_HASH) not in (None, before_hash)
        and meta.get(hr_demo.META_VERSION) == before_version
    )

hr_demo.wait_until(acl_written, hr_demo.WRITE_WAIT, "acl snapshot")
show("acl")


acl               v2 hr_policy_admin          v2 hr_policy_admin versions=3


## Delete


In [6]:
hr_demo.admin("DELETE", f"/admin/policies/{POLICY_ID}")
missing = hr_demo.expect_status("GET", f"/policies/{POLICY_ID}", 404)
print("source", missing.status_code)
hr_demo.wait_until(
    lambda: hr_demo.versions(KEY) == 0,
    hr_demo.DELETE_WAIT,
    "delete sweep",
)
show("delete")


source 404
delete            404                         missing


## Drift


In [7]:
def aligned() -> bool:
    return len(hr_demo.list_keys("raw/hr/")) == hr_demo.source_total()

hr_demo.wait_until(aligned, hr_demo.DELETE_WAIT, "drift back to zero")
snapshots = len(hr_demo.list_keys("raw/hr/"))
source = hr_demo.source_total()
drift = snapshots - source
rows.append(f"{'drift':<18}{source:<28}{snapshots} drift {drift}")
print("\n".join(rows))
assert drift == 0


step              source                      snapshot
create            v1 hr_employee              v1 hr_employee versions=1
update            v2 hr_employee              v2 hr_employee versions=2
acl               v2 hr_policy_admin          v2 hr_policy_admin versions=3
delete            404                         missing
drift             286                         286 drift 0


## Cleanup


In [8]:
hr_demo.delete_if_present("policy", POLICY_ID)
hr_demo.wait_until(
    lambda: hr_demo.versions(KEY) == 0,
    hr_demo.DELETE_WAIT,
    "cleanup delete",
)
print("removed", KEY)


removed raw/hr/policy/pol-demo-cycle.md
